In [2]:
# ═══════════════════════════════════════════════════════════════
#  TRPG 调查员助手 —— 主流程 Notebook
# ═══════════════════════════════════════════════════════════════

import sys
import json
from datetime import datetime
from IPython.display import HTML, display

# 将 src/ 加入路径以导入依赖模块
sys.path.insert(0, "../src")

from scenario_core import DirectedGraph, ScenarioWorld
from llm import call_deepseek, set_llm_log_file
from prompts import build_narrative_prompt, set_prompt_log_file, parse_narrative_output
from game_loop import handle_user_input, _handle_spawn_command, _apply_side_effects
from library import WeaponLibrary, EnemyLibrary, ContentInjector
from trpg_display import (
    display_narrative, display_scene, display_system, display_debug,
    display_input_area, render_scene_to_html, display_split_result,
)

# ── COC 7th 车卡系统（替代旧 Player 类）──
# 调查员通过前端 character.html 创建并导出 JSON，这里负责加载。
# 如果还没有创建角色卡，运行时会自动生成一个默认调查员。
from investigator import Investigator, load_investigator
from investigator.rules import roll_stats, calc_derived, create_skill_list

In [3]:
# ═══════════════════════════════════════════════════════════════
#  Prompt 日志配置
# ═══════════════════════════════════════════════════════════════

PROMPT_LOG_FILE = f"../logs/prompt_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
set_prompt_log_file(PROMPT_LOG_FILE)
set_llm_log_file(PROMPT_LOG_FILE)

In [4]:
# ============================================================
#  武器/敌人库初始化（新增 — parser 系统升级）
# ============================================================

weapon_lib = WeaponLibrary()
weapon_lib.load_core()
enemy_lib = EnemyLibrary()
enemy_lib.load_core()
injector = ContentInjector(weapon_lib, enemy_lib)
display_system(
    f"武器库：{len(weapon_lib)} 件 | 敌人库：{len(enemy_lib)} 个 | "
    f"注入器：{"就绪" if injector else "未初始化"}",
    "info"
)

In [ ]:
def run_game(character_path: str = None):
    """
    启动 TRPG 游戏主循环。

    参数:
        character_path: 调查员 JSON 文件路径（可选）。
                       如果为 None，自动查找 ../investigator/test_character.json；
                       如果文件不存在，使用默认掷骰生成的调查员。
    """
    import json as _json
    import os as _os

    # ── 从 L2 keeper JSON 加载模组数据（新三层格式）──
    with open("../data/modules/常暗之厢/l2_keeper.json", "r", encoding="utf-8") as f:
        l2 = _json.load(f)
    scenes = l2["scenes"]
    print(type(scenes))
    events = l2["events"]
    try:
        with open("../data/abstract.txt", "r", encoding="utf-8") as f:
            abstract = f.read()
    except FileNotFoundError:
        abstract = ""  # abstract 可选，L3.driving_force 可替代

    # ── 构建世界 ──
    graph = DirectedGraph(scenes=scenes, events=events)
    world = ScenarioWorld(graph, start_node="6号车厢",
                          background_story=abstract)

    # ── 加载调查员（COC 7th 车卡系统）──
    # 优先从 JSON 文件加载（前端 character.html 导出）；
    # 如果文件不存在，自动掷骰生成一个默认调查员。
    if character_path is None:
        character_path = "../investigator/test_character.json"

    if _os.path.exists(character_path):
        # ── 方式 1：从 JSON 文件加载 ──
        investigator = load_investigator(character_path)
        display_system(
            f"已加载调查员：{investigator.name} | "
            f"职业：{investigator.occupation.name if investigator.occupation else '无'} | "
            f"HP={investigator.derived.HP} SAN={investigator.derived.SAN}",
            "info"
        )
    else:
        # ── 方式 2：掷骰生成默认调查员（无 JSON 时的 fallback）──
        display_system(
            f"未找到角色卡文件 {character_path}，正在掷骰生成默认调查员...",
            "warn"
        )
        investigator = Investigator(name="调查员A", age=25, gender="男")
        investigator.stats = roll_stats()
        investigator.skills = create_skill_list()
        investigator.derived = calc_derived(investigator.stats, investigator.age)
        display_system(
            f"已生成调查员：{investigator.name} | "
            f"HP={investigator.derived.HP} SAN={investigator.derived.SAN}",
            "info"
        )

    # 注入世界（Investigator 完全替代旧 Player 类）
    world.set_player(investigator)

    # ── 确保存档目录存在 ──
    _os.makedirs("../data/saves", exist_ok=True)

    # ── 存档提示格式化 ──
    def _save_banner(slot, path):
        return (
            f"╔══════════════════════════════════════╗\n"
            f"║  ◆ 游戏存档  [{slot}]\n"
            f"╠══════════════════════════════════════╣\n"
            f"║  TURN {world.memory.turn:<4}  位置：{world.current_location}\n"
            f"║  路径：{path}\n"
            f"╚══════════════════════════════════════╝"
        )

    def _char_banner(action, path):
        inv = world.player
        name = inv.name if inv else "—"
        hp = inv.derived.HP if inv else "—"
        san = inv.derived.SAN if inv else "—"
        return (
            f"╔══════════════════════════════════════╗\n"
            f"║  ◆ 调查员{action}                    ║\n"
            f"╠══════════════════════════════════════╣\n"
            f"║  姓名：{name:<30} ║\n"
            f"║  HP={hp:<3}  SAN={san:<3}"
            f"                        ║\n"
            f"║  路径：{path}\n"
            f"╚══════════════════════════════════════╝"
        )

    turn = 0

    # ── 开场 ──
    display_system("游戏开始。输入 /help 查看可用命令。", "info")
    display(HTML(render_scene_to_html(world)))

    try:
        # call_deepseek(json_mode=False) → temperature=0.7, max_tokens=20000
        initial_narrative = call_deepseek(
            build_narrative_prompt(
                world,
                user_input="（游戏开始）",
                action_result="（从沉睡中醒来，环顾四周）",
                events_result="",
            ),
            json_mode=False
        )